In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

In [3]:
df = pd.read_csv("S&P 5 year Data.csv", parse_dates = ["Date"])
df.set_index("Date" , inplace = True)
df.rename(columns={"Close/Last": "Close"}, inplace=True)

print(df.head())

              Close     Open     High      Low
Date                                          
2025-04-02  5670.97  5580.76  5695.31  5571.48
2025-04-01  5633.07  5597.53  5650.57  5558.52
2025-03-31  5611.85  5527.91  5627.56  5488.73
2025-03-28  5580.94  5679.20  5685.89  5572.42
2025-03-27  5693.31  5695.64  5732.28  5670.94


In [4]:
print(df.isnull().sum())

Close    0
Open     0
High     0
Low      0
dtype: int64


In [5]:
#Reverse the order of data
df = df.iloc[::-1]
df.head()

,Close,Open,High,Low
Date,,,,
2020-04-03,2488.65,2514.92,2538.18,2459.96
2020-04-06,2663.68,2578.28,2676.85,2574.57
2020-04-07,2659.41,2738.65,2756.89,2657.67
2020-04-08,2749.98,2685.00,2760.75,2663.30
2020-04-09,2789.82,2776.99,2818.57,2762.36


In [6]:
#Feature engineering


#SMA = Pt/n where N = window size, or number of days, 
#Simple Moving Average (SMA) is a technical indicator used in financial markets to analyze trends 
#in stock prices by smoothing out short-term fluctuations.
def calculate_sma(prices, period):
    sma_values = []
    for i in range(len(prices)):
        if i< period - 1:
            sma_values.append(None)
        else:
            sma = np.mean(prices[i-period + 1 : i +1])
            sma_values.append(sma)
    return sma_values
df["SMA_5"] = df["Close"].rolling(window=5, min_periods=5).mean()
df["SMA_10"] = df["Close"].rolling(window=10, min_periods=10).mean()
#Long term
df["SMA_50"] = df["Close"].rolling(window=50, min_periods=50).mean()
df["SMA_100"] = df["Close"].rolling(window=100, min_periods=100).mean()
df["SMA_100"].head(101)

Date
2020-04-03          NaN
2020-04-06          NaN
2020-04-07          NaN
2020-04-08          NaN
2020-04-09          NaN
                ...    
2020-08-20          NaN
2020-08-21          NaN
2020-08-24          NaN
2020-08-25    3075.2085
2020-08-26    3085.1093
Name: SMA_100, Length: 101, dtype: float64

In [7]:
#Exponential Moving Average : puts more value on recent values
def calculate_ema(prices, period):
    ema_values = []
    alpha = 2 / (period + 1)
    ema_prev = None

    for i in range(len(prices)):
        price = prices[i]

        # Skip if price is None or NaN
        if price is None or np.isnan(price):
            ema_values.append(None)
            continue

        if i < period - 1:
            ema_values.append(None)
        elif i == period - 1:
            valid_prices = [p for p in prices[:period] if p is not None and not np.isnan(p)]
            sma = np.mean(valid_prices)
            ema_values.append(sma)
            ema_prev = sma
        else:
            if ema_prev is None or np.isnan(ema_prev):
                ema_values.append(None)
                continue
            ema = alpha * price + (1 - alpha) * ema_prev
            ema_values.append(ema)
            ema_prev = ema

    return ema_values

df['EMA_12'] = calculate_ema(df['Close'].tolist(), 12)
df['EMA_26'] = calculate_ema(df['Close'].tolist(), 26)

# Drop rows where EMA_12 or EMA_26 is not available
df.dropna(subset=['EMA_12', 'EMA_26'], inplace=True)

# MACD = EMA 12 - 26 Shows the trend direction and momentum — is the short-term EMA rising faster than the long-term one
df['MACD_Line'] = df['EMA_12'] - df['EMA_26']
#The signal line is used as a trigger for buy and sell signals, with a bullish signal (potential buy) occurring when the 


#MACD line crosses above the signal line and a bearish signal (potential sell) when the MACD line crosses below the signal line
#Signal line is a smoothed version of the MACD Line — used to generate buy/sell signals
df['Signal_Line'] = calculate_ema(df['MACD_Line'].tolist(), 9)

#MACD histogram measures the strength of the momentum and whether it’s increasing or weakening. MACD> Signal means it is bullish(buy) upward momentum built

df['MACD_Histogram'] = df['MACD_Line'] - df['Signal_Line']

df[['EMA_12', 'EMA_26', 'MACD_Line', 'Signal_Line','MACD_Histogram']].head(15)
df.head(50)

,Close,Open,High,Low,SMA_5,SMA_10,SMA_50,SMA_100,EMA_12,EMA_26,MACD_Line,Signal_Line,MACD_Histogram
Date,,,,,,,,,,,,,
2020-05-11,2930.32,2915.46,2944.25,2903.44,2891.634,2884.695,NaN,NaN,2871.668184,2812.911538,58.756645,NaN,NaN
2020-05-12,2870.12,2939.50,2945.82,2869.59,2891.970,2885.368,NaN,NaN,2871.430002,2817.149202,54.280799,NaN,NaN
2020-05-13,2820.00,2865.86,2874.14,2793.15,2886.286,2873.417,NaN,NaN,2863.517694,2817.360372,46.157321,NaN,NaN
2020-05-14,2852.50,2794.54,2852.80,2766.64,2880.548,2867.424,NaN,NaN,2861.822664,2819.963308,41.859356,NaN,NaN
2020-05-15,2863.70,2829.95,2865.01,2816.78,2867.328,2870.723,NaN,NaN,2862.111485,2823.203063,38.908422,NaN,NaN
2020-05-18,2953.91,2913.86,2968.09,2913.86,2872.046,2881.840,NaN,NaN,2876.234333,2832.885058,43.349275,NaN,NaN
2020-05-19,2922.94,2948.59,2964.21,2922.35,2882.610,2887.290,NaN,NaN,2883.419821,2839.555795,43.864026,NaN,NaN
2020-05-20,2971.61,2953.63,2980.29,2953.63,2912.932,2899.609,NaN,NaN,2896.987540,2849.337588,47.649953,NaN,NaN
2020-05-21,2948.51,2969.95,2978.50,2938.57,2932.134,2906.341,NaN,NaN,2904.914073,2856.683692,48.230380,47.006242,1.224138


In [8]:
def calculate_rsi(prices, period=14):
    # Ensure prices are numeric and clean
    prices = pd.to_numeric(prices, errors='coerce').ffill()

    delta = prices.diff()
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)

    # Calculate rolling mean for gains and losses
    avg_gain = pd.Series(gain, index=prices.index).rolling(window=period).mean()
    avg_loss = pd.Series(loss, index=prices.index).rolling(window=period).mean()

    # Prevent division by zero
    rs = np.where(avg_loss == 0, 100, avg_gain / (avg_loss + 1e-10))
    rsi = 100 - (100 / (1 + rs))

    return pd.Series(rsi, index=prices.index)
#RSI is zero if average gain is zero

# Bollinger Bands Calculation Function
def calculate_bollinger_bands(prices, window=20, num_std=2):
    sma = prices.rolling(window=window).mean()
    std = prices.rolling(window=window).std()
#STD = 2 SD (Default): Works well in most market conditions. 
#STD= 3 : Ideal for highly volatile markets or large-cap stocks where volatility is more extreme 
    upper_band = sma + (num_std * std)   #upper = SMA 20  + 2 std
    lower_band = sma - (num_std * std)   #lower = SMA 20 - 2 std
    #Middle = SMA 20
    return sma, upper_band, lower_band
#Trading signal can be not accurate
# lower std will have less accuracy.

close_prices = df["Close"]

# Calculate RSI
df["RSI_14"] = calculate_rsi(close_prices)

# Calculate Bollinger Bands
df["BB_Middle"], df["BB_Upper"], df["BB_Lower"] = calculate_bollinger_bands(close_prices)

In [9]:
# Trim the dataset to only include rows from '2020-08-25' onward because SMA_100 has value
df = df[df.index >= '2020-08-25']

# Display the first few rows of the cleaned dataset
df.head()


,Close,Open,High,Low,SMA_5,SMA_10,SMA_50,SMA_100,EMA_12,EMA_26,MACD_Line,Signal_Line,MACD_Histogram,RSI_14,BB_Middle,BB_Upper,BB_Lower
Date,,,,,,,,,,,,,,,,,
2020-08-25,3443.62,3435.95,3444.21,3425.84,3406.484,3393.082,3236.3812,3075.2085,3384.577679,3330.560423,54.017256,52.416113,1.601143,77.033649,3351.5045,3459.338292,3243.670708
2020-08-26,3478.73,3449.97,3481.07,3444.15,3427.260,3402.920,3243.4610,3085.1093,3399.062652,3341.535947,57.526704,53.438232,4.088473,78.415720,3362.5190,3475.220323,3249.817677
2020-08-27,3484.55,3485.14,3501.38,3468.35,3447.068,3414.032,3250.8822,3093.3180,3412.214551,3352.129581,60.084971,54.767579,5.317391,78.760413,3374.4355,3485.751638,3263.119362
2020-08-28,3508.01,3494.69,3509.23,3484.32,3469.238,3427.548,3258.7356,3101.8040,3426.952313,3363.676279,63.276034,56.469270,6.806764,79.992682,3386.2800,3501.647425,3270.912575
2020-08-31,3500.31,3509.73,3514.77,3493.25,3483.044,3439.380,3266.7870,3109.3073,3438.238111,3373.797295,64.440816,58.063579,6.377236,86.719852,3396.5650,3514.176878,3278.953122


In [15]:
#CTA commodity trade advisors. Hedge fund.
#Brevon howard
#Momentum.
#CAMpbell dwshaw, millenium.  back test for return
#Quant finance
#Arrow street

#LSTM
from statsmodels.tsa.stattools import adfuller

# Perform the Augmented Dickey-Fuller (ADF) Test
adf_test = adfuller(df['Close'])
# Display the results
adf_results = {
    'ADF Statistic': adf_test[0],
    'p-value': adf_test[1],
    'Critical Values': adf_test[4]
}
adf_results
# result shows it is fail to reject so data is non-stationary.

{'ADF Statistic': -1.00747975120104,
 'p-value': 0.7505264670247765,
 'Critical Values': {'1%': -3.4360194465416387,
  '5%': -2.8640434537995523,
  '10%': -2.5681028978640104}}

In [16]:
#Training model
#Goal is to predict close price for S&P
X = df.drop(columns=['Close'])
y = df['Close'].shift(-1) #Given today's signal and predict tomorrow's price
X = X.iloc[:-1]
y = y.iloc[:-1]


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False  # crucial for time series
)
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)  
X_test_scaled = scaler.transform(X_test)
X_train_scaled.shape, X_test_scaled.shape

((924, 16), (232, 16))

In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import r2_score
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred = lr_model.predict(X_test_scaled)


mse = MSE(y_test, y_pred)
rmse = np.sqrt(MSE(y_test, y_pred, squared=False))
mae = MAE(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

mse, rmse, mae, r2


D:\anaconda\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


(2633.103690907309, 7.163363512617905, 39.12822856911238, 0.9648015936260226)

In [22]:
lr_model.intercept_ , lr_model.coef_

(3023.2933444003706,
 array([-1630.7388553 ,  1033.47767136,  1240.11970958, -1760.15098095,
         -551.63890109,  -163.16655886,   -85.12201495,  1281.2698206 ,
         1290.56475894,   209.01809742,    71.74303709,   380.52474699,
           26.03335271,   391.91494279,   354.7317269 ,   405.97765521]))

In [23]:
X_test_scaled[-1]  # Next day
lr_model.predict([X_test_scaled[-1]])  # prediction


array([5624.54732478])

In [24]:
X_test_scaled[-2]  # Next day
lr_model.predict([X_test_scaled[-2]])


array([5613.98835077])

In [25]:
X_test_scaled[-2]  # Next day
lr_model.predict([X_test_scaled[-2]])

array([5613.98835077])